In [1]:
import pandas as pd
from sklearn.metrics import cohen_kappa_score, accuracy_score

In [ ]:
# Load original inspection set and the IRR sample
inspection_set_annotated = pd.read_csv("../../data/processed/inspection_set.csv")
bayflood_irr = pd.read_csv("../../data/revisions/irr/bayflood_irr.csv")

In [3]:
bayflood_irr['image'].values[0]

'/data/local-files/?d=/share/ju/nexar_data/training_datasets/street_flooding/all_no_letterboxing/nlbx_ccaafd467c010e108c1524696f94a7da.jpg'

In [4]:
# Preprocess bayflood_irr
bayflood_irr = bayflood_irr.iloc[100:].copy()
bayflood_irr['choice'] = bayflood_irr['choice'].astype(str).str.strip()
bayflood_irr = bayflood_irr[bayflood_irr['choice'].isin(["Flooded", "Not Flooded"])]
bayflood_irr['label_irr'] = bayflood_irr['choice'].map({"Flooded": 1, "Not Flooded": 0})

In [5]:
inspection_set_annotated['frame_id'] = inspection_set_annotated['image'].str.replace('/data/local-files/?d=/share/XXXX-19/nexar_data/training_datasets/street_flooding/all_no_letterboxing/', '').str.replace('.jpg', '')

In [6]:
bayflood_irr['frame_id'] = bayflood_irr['image'].str.replace('/data/local-files/?d=/share/ju/nexar_data/training_datasets/street_flooding/all_no_letterboxing/', '').str.replace('.jpg', '')

In [8]:
# Merge to find overlapping images and filter out unannotated ones
merged_df = pd.merge(
    bayflood_irr[['frame_id', 'label_irr']], 
    inspection_set_annotated[['frame_id', 'sentiment_1']], 
    on='frame_id', 
    how='inner'
)

In [9]:
# Calculate metrics
kappa = cohen_kappa_score(merged_df['label_irr'], merged_df['sentiment_1'])
accuracy = accuracy_score(merged_df['label_irr'], merged_df['sentiment_1'])

print(f"Number of overlapping images: {len(merged_df)}")
print(f"Cohen's Kappa: {kappa:.4f}")
print(f"Accuracy: {accuracy:.4f}")

Number of overlapping images: 400
Cohen's Kappa: 0.6195
Accuracy: 0.8050


In [10]:
# Confusion Matrix
ct = pd.crosstab(merged_df['label_irr'], merged_df['sentiment_1'], rownames=['Annotator 2 (IRR)'], colnames=['Annotator 1 (Original)'])
ct

Annotator 1 (Original),0,1
Annotator 2 (IRR),,
0,146,74
1,4,176


In [11]:
# Conditional Probabilities
# Row index is label_irr (You), Column index is sentiment_1 (Other)

p_other_flooded_given_you_flooded = ct.loc[1, 1] / ct.loc[1].sum()
p_other_not_flooded_given_you_not_flooded = ct.loc[0, 0] / ct.loc[0].sum()

print(f"P(Other = Flooded | You = Flooded): {p_other_flooded_given_you_flooded:.4f}")
print(f"P(Other = Not Flooded | You = Not Flooded): {p_other_not_flooded_given_you_not_flooded:.4f}")

P(Other = Flooded | You = Flooded): 0.9778
P(Other = Not Flooded | You = Not Flooded): 0.6636
